In [6]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

cm_files = sorted(Path("output_files").glob("confusion_matrices_*.csv"))

datasets = {}
for f in cm_files:
    label = f.stem.replace("confusion_matrices_", "")
    df = pd.read_csv(f)
    cm_cols = [c for c in df.columns if c.startswith("true_") and "_pred_" in c]
    df[cm_cols] = df[cm_cols].div(df[cm_cols].sum(axis=1), axis=0)
    datasets[label] = (df, cm_cols)


def x_values_for_plot(df: pd.DataFrame, label: str):
    """X-axis values per dataset: Apple uses chunk_idx; COVID uses MM-DD + year 2020; else timestamp or chunk_idx."""
    if label.startswith("Apple-Twitter"):
        return df["chunk_idx"]
    if label == "global_covid19_tweets":
        s = df["timestamp"].astype(str).str.strip()
        return pd.to_datetime(s + "-2020", format="%m-%d-%Y", utc=True, errors="coerce")
    if "timestamp" in df.columns and df["timestamp"].notna().any():
        return pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    return df["chunk_idx"]


def uses_date_xaxis(df: pd.DataFrame, label: str) -> bool:
    if label.startswith("Apple-Twitter"):
        return False
    if label == "global_covid19_tweets":
        return True
    if "timestamp" not in df.columns or not df["timestamp"].notna().any():
        return False
    t = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    return bool(t.notna().any())


fig = go.Figure()
labels = list(datasets.keys())
trace_counts = []

for label, (df, cm_cols) in datasets.items():
    x = x_values_for_plot(df, label)
    for col in cm_cols:
        fig.add_trace(go.Scatter(
            x=x,
            y=df[col],
            mode="lines+markers",
            name=col,
            visible=(label == labels[0]),
        ))
    trace_counts.append(len(cm_cols))

buttons = []
for i, label in enumerate(labels):
    visibility = []
    for j, count in enumerate(trace_counts):
        visibility.extend([i == j] * count)
    df_i = datasets[label][0]
    date_x = uses_date_xaxis(df_i, label)
    layout_patch = {"title": f"Confusion Matrix Values Over Time — {label}"}
    if date_x:
        layout_patch["xaxis"] = {"type": "date", "title": {"text": "Timestamp (from confusion matrix report)"}}
    else:
        layout_patch["xaxis"] = {"type": "linear", "title": {"text": "Chunk index (no timestamp in file)"}}
    buttons.append(dict(label=label, method="update", args=[
        {"visible": visibility},
        layout_patch,
    ]))

df0 = datasets[labels[0]][0]
label0 = labels[0]
xaxis0 = (
    {"type": "date", "title": {"text": "Timestamp (from confusion matrix report)"}}
    if uses_date_xaxis(df0, label0)
    else {"type": "linear", "title": {"text": "Chunk index (no timestamp in file)"}}
)

fig.update_layout(
    title=f"Confusion Matrix Values Over Time — {labels[0]}",
    yaxis_title="Proportion",
    legend_title="CM Cell",
    hovermode="x unified",
    template="plotly_white",
    xaxis=xaxis0,
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.0,
        xanchor="left",
        y=1.15,
        yanchor="top",
    )],
)
fig.show()